## 1. ライブラリのインポート

In [81]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

import warnings
warnings.filterwarnings('ignore')

# Plotting settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

In [82]:
n_estimators = 281
max_depth = 6

def evaluate_xgb(X_data, y_data, n_estimators=n_estimators, max_depth=max_depth, cv=5, use_gpu=False):
    """
    XGBoostでクロスバリデーション評価を行う関数
    
    Parameters:
    -----------
    X_data : DataFrame
        特徴量データ
    y_data : Series
        ターゲットデータ
    n_estimators : int
        木の数（デフォルト: 281）
    max_depth : int
        木の深さ（デフォルト: 6）
    cv : int
        クロスバリデーションのfold数（デフォルト: 5）
    use_gpu : bool
        GPUを使用するか（デフォルト: False）
    
    Returns:
    --------
    dict : CV結果を含む辞書
    """
    # モデル作成
    params = {
        'n_estimators': n_estimators,
        'max_depth': max_depth,
        'random_state': 42,
        'eval_metric': 'logloss',
        'n_jobs': -1
    }
    
    if use_gpu:
        params['tree_method'] = 'gpu_hist'
        params['device'] = 'cuda'
    
    model = XGBClassifier(**params)
    
    # クロスバリデーション（Accuracy）
    cv_scores_acc = cross_val_score(model, X_data, y_data, cv=cv, scoring='accuracy', n_jobs=-1)
    
    # クロスバリデーション（AUC）
    cv_scores_auc = cross_val_score(model, X_data, y_data, cv=cv, scoring='roc_auc', n_jobs=-1)
    
    # 結果表示
    #print("="*60)
    #print(f"XGBoost CV Results (GPU: {use_gpu})")
    #print("="*60)
    #print(f"Features: {X_data.shape[1]}")
    #print(f"Samples: {X_data.shape[0]}")
    #print(f"CV Folds: {cv}")
    #print(f"n_estimators: {n_estimators}, max_depth: {max_depth}")
    #print(f"\nAccuracy CV Scores: {[f'{s:.4f}' for s in cv_scores_acc]}")
    #print(f"Accuracy Mean: {cv_scores_acc.mean():.4f}")
    #print(f"Accuracy Std:  {cv_scores_acc.std():.4f}")
    #print(f"\nAUC CV Scores: {[f'{s:.4f}' for s in cv_scores_auc]}")
    #print(f"AUC Mean: {cv_scores_auc.mean():.4f}")
    #print(f"AUC Std:  {cv_scores_auc.std():.4f}")
    #print("="*60)
    
    return {
        'accuracy_mean': cv_scores_acc.mean(),
        'accuracy_std': cv_scores_acc.std(),
        'accuracy_scores': cv_scores_acc,
        'auc_mean': cv_scores_auc.mean(),
        'auc_std': cv_scores_auc.std(),
        'auc_scores': cv_scores_auc,
        'model_params': params
    }


# （任意）全木dumpをファイルに保存だけしたい場合
from xgboost import XGBClassifier
from pathlib import Path

def save_all_xgb_tree_dumps(
    X_data,
    y_data,
    n_estimators=n_estimators,
    max_depth=max_depth,
    with_stats=True,
    out_file=f"xgb_tree_dump_all_n_estimators{n_estimators}.txt"
):
    model = XGBClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42,
        eval_metric='logloss',
        n_jobs=-1,
    )
    model.fit(X_data, y_data)

    booster = model.get_booster()
    dumps = booster.get_dump(with_stats=with_stats)

    blocks = []
    for i, d in enumerate(dumps):
        blocks.append(f"===== Tree {i} =====\n{d}")

    out_path = Path(out_file)
    out_path.write_text("\n\n".join(blocks), encoding="utf-8")
    print(f"Saved {len(dumps)} trees to: {out_path.resolve()}")
    return model, out_path


## 2. データの読み込み

In [83]:
# Load data
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

# Separate features and target
X = train.drop(['id', 'diagnosed_diabetes'], axis=1)
y = train['diagnosed_diabetes']
X_test = test.drop('id', axis=1)

print(f"\nTarget distribution:\n{y.value_counts(normalize=True)}")



Train shape: (700000, 26)
Test shape: (300000, 25)

Target distribution:
diagnosed_diabetes
1.0    0.623296
0.0    0.376704
Name: proportion, dtype: float64


## 3. ベースラインの評価

In [84]:
# Identify categorical columns
categorical_cols = X.select_dtypes(include='object').columns.tolist()
print(f"Categorical columns: {categorical_cols}")

# Label encoding for categorical variables
# コピーを作成して、元のデータを保持
X_encoded = X.copy()
X_test_encoded = X_test.copy()

label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X[col])
    X_test_encoded[col] = le.transform(X_test[col])
    label_encoders[col] = le

print("\nLabel encoding completed!")
print(f"Encoded data shape: {X_encoded.shape}")


Categorical columns: ['gender', 'ethnicity', 'education_level', 'income_level', 'smoking_status', 'employment_status']

Label encoding completed!
Encoded data shape: (700000, 24)


## 4. 新しい特徴量の作成

In [85]:
#エンコード後のデータをコピー
X_fe = X_encoded.copy()
X_test_fe = X_test_encoded.copy()   
#####################
#####################
#元特徴量の変換
#####################
#####################

#log1p変換
X_fe['physical_activity_minutes_per_week'] = np.log1p(X_fe['physical_activity_minutes_per_week'])
X_test_fe['physical_activity_minutes_per_week'] = np.log1p(X_test_fe['physical_activity_minutes_per_week'])
print(f"Total features now: {X_fe.shape[1]}")


Total features now: 24


## 3.5 評価関数の評価

In [86]:
# ===== 特徴量追加前後を evaluate_xgb で評価（パラメータはデフォルトのまま） =====
print("=" * 60)
print("特徴量追加前後の評価（evaluate_xgb）")
print("=" * 60)
print(f"Original features: {X_encoded.shape[1]}")
print(f"Engineered features: {X_fe.shape[1]}")
print(f"Added: {X_fe.shape[1] - X_encoded.shape[1]}")

res_original = evaluate_xgb(X_encoded, y)
res_fe = evaluate_xgb(X_fe, y)

print("\n--- Summary ---")
print(f"Original AUC mean: {res_original['auc_mean']:.4f}  | Acc mean: {res_original['accuracy_mean']:.4f}")
print(f"FE       AUC mean: {res_fe['auc_mean']:.4f}  | Acc mean: {res_fe['accuracy_mean']:.4f}")
print("=" * 60)


特徴量追加前後の評価（evaluate_xgb）
Original features: 24
Engineered features: 24
Added: 0

--- Summary ---
Original AUC mean: 0.7203  | Acc mean: 0.6803
FE       AUC mean: 0.7203  | Acc mean: 0.6803


## →前処理無しが一番良い結果では？アンサンブルを試してみる

## 没になったアイデア

### ・physical_activity_minutes_per_weekのlog1p変換したが、cvは改善しなかった
### ・新しい特徴量を作ったが、cvは改善しなかった

In [87]:
## 4.1 健康リスク指標（シンプルに追加していく）
## ベースラインは触らず、X_encoded から派生特徴量を作る

#
## 列名ゆれ対策（ある方を使う）
#bmi_col = "bmi" if "bmi" in X_fe.columns else ("body_mass_index" if "body_mass_index" in X_fe.columns else None)
#chol_total_col = (
#    "cholesterol_total"
#    if "cholesterol_total" in X_fe.columns
#    else ("total_cholesterol" if "total_cholesterol" in X_fe.columns else None)
#)
#
#added = []
#
## 血圧系
#
#X_fe["bp_ratio"] = X_fe["systolic_bp"] / (X_fe["diastolic_bp"] + 1e-6)
#X_test_fe["bp_ratio"] = X_test_fe["systolic_bp"] / (X_test_fe["diastolic_bp"] + 1e-6)
#added.append("bp_ratio")
#
#X_fe["bp_mean"] = (X_fe["systolic_bp"] + X_fe["diastolic_bp"]) / 2
#X_test_fe["bp_mean"] = (X_test_fe["systolic_bp"] + X_test_fe["diastolic_bp"]) / 2
#added.append("bp_mean")
#
## 脂質系
#
#X_fe["cholesterol_ratio"] = X_fe[chol_total_col] / (X_fe["hdl_cholesterol"] + 1e-6)
#X_test_fe["cholesterol_ratio"] = X_test_fe[chol_total_col] / (X_test_fe["hdl_cholesterol"] + 1e-6)
#added.append("cholesterol_ratio")
#
#X_fe["non_hdl_cholesterol"] = X_fe[chol_total_col] - X_fe["hdl_cholesterol"]
#X_test_fe["non_hdl_cholesterol"] = X_test_fe[chol_total_col] - X_test_fe["hdl_cholesterol"]
#added.append("non_hdl_cholesterol")
#
#
#X_fe["triglyceride_hdl_ratio"] = X_fe["triglycerides"] / (X_fe["hdl_cholesterol"] + 1e-6)
#X_test_fe["triglyceride_hdl_ratio"] = X_test_fe["triglycerides"] / (X_test_fe["hdl_cholesterol"] + 1e-6)
#added.append("triglyceride_hdl_ratio")
#
#
#X_fe["ldl_hdl_ratio"] = X_fe["ldl_cholesterol"] / (X_fe["hdl_cholesterol"] + 1e-6)
#X_test_fe["ldl_hdl_ratio"] = X_test_fe["ldl_cholesterol"] / (X_test_fe["hdl_cholesterol"] + 1e-6)
#added.append("ldl_hdl_ratio")
#
###################################
###################################
#
#
#X_fe["gender_smoking"] = X_fe["gender"] * X_fe["smoking_status"]
#X_test_fe["gender_smoking"] = X_test_fe["gender"] * X_test_fe["smoking_status"]
#added.append("gender_smoking")
#
#
#X_fe["age_family_history"] = X_fe["age"] * X_fe["family_history_diabetes"]
#X_test_fe["age_family_history"] = X_test_fe["age"] * X_test_fe["family_history_diabetes"]
#added.append("age_family_history")
#
#pa_col = (
#"physical_activity_minutes_per_week"
#if "physical_activity_minutes_per_week" in X_fe.columns
#else ("physical_activity_hours_per_week" if "physical_activity_hours_per_week" in X_fe.columns else None)
#)
#
#X_fe["bmi_activity"] = X_fe[bmi_col] * X_fe[pa_col]
#X_test_fe["bmi_activity"] = X_test_fe[bmi_col] * X_test_fe[pa_col]
#added.append("bmi_activity")
#
#
#X_fe["age_alcohol"] = X_fe["age"] * X_fe["alcohol_consumption_per_week"]
#X_test_fe["age_alcohol"] = X_test_fe["age"] * X_test_fe["alcohol_consumption_per_week"]
#added.append("age_alcohol")